# Demo: Mensa Meal Planner

This notebook loads the bundled `examples/demo.json` file, explores the typed menu objects and helper utilities, and builds meal plans with the integer linear programming API. Install the project in editable mode from its repository root with `uv pip install -e .` before opening the notebook.

In [1]:
from mensa.ilp import UserPreferences, solve_meal_plan
from mensa.loader import load_menu
from mensa.model import (
    describe_substance,
    expand_substance_codes,
    split_substance_codes,
)

## Load the bundled menu

`load_menu` accepts a path to the same normalized JSON format used by the CLI. It returns a `LoadedMenu` with typed canteens and meals. The data is grouped for convenience lookup.

In [2]:
menu = load_menu("./demo.json")
meals = list(menu.meals)
menu_dates = sorted(menu.meals_by_date)

print(f"Canteens: {', '.join(canteen.name for canteen in menu.canteens)}")
print(f"Meals: {len(meals)} | Dates: {', '.join(day.isoformat() for day in menu_dates)}")
for day, day_meals in sorted(menu.meals_by_date.items()):
    kinds = {kind: sum(meal.kind.value == kind for meal in day_meals) for kind in ("main", "breakfast", "side")}
    print(f"{day}: {len(day_meals)} meals | {kinds}")

Canteens: Galerie
Meals: 59 | Dates: 2026-09-29, 2026-09-30, 2026-10-01
2026-09-29: 21 meals | {'main': 21, 'breakfast': 0, 'side': 0}
2026-09-30: 21 meals | {'main': 21, 'breakfast': 0, 'side': 0}
2026-10-01: 17 meals | {'main': 17, 'breakfast': 0, 'side': 0}


`LoadedMenu` exposes date and ID lookups. Each `Meal` provides role-based pricing, diet classification, climate scoring, allergen/additive separation, and a JSON-ready `to_dict()` representation. Substance helpers normalize code lists and expand grouped allergen codes such as `20` into their known variants.

In [ ]:
sample_meal = meals[0]

print(f"Canteens by ID: {menu.canteens_by_id}")

print("Inspecting the sample meal")
print(f"Sample: {sample_meal.name} ({sample_meal.diet_category.value})")
print(f"Student / staff / guest: {[sample_meal.price_for(role) for role in ('student', 'staff', 'guest')]}")
print(f"Climate rating / score: {sample_meal.climate_rating} / {sample_meal.climate_score}")
print(f"Allergens: {sample_meal.allergen_codes} | Additives: {sample_meal.additive_codes}")
print(f"JSON-ready fields: {list(sample_meal.to_dict())}")

Inspecting a meal:
Canteens by ID: {451: Canteen(id=451, name='Galerie')}
Sample: Hamburger Rind Menü (meat)
Student / staff / guest: [6.5, 8.6, 9.7]
Climate rating / score: B / 1
Allergens: ('20a', '20b', '20c', '26', '29', '30') | Additives: ('1', '2', '4', '7')
JSON-ready fields: ['id', 'date', 'name', 'canteen_id', 'canteen', 'section', 'diet', 'diet_category', 'components', 'allergens', 'additives', 'price', 'climate_rating', 'notes']


In [4]:
print("Functions for allergens/additives")
raw_codes = "20a, 26; 4"
print(f"Normalized codes: {split_substance_codes(raw_codes)}")
print(f"Expanded gluten group 20: {sorted(expand_substance_codes(['20']))}")
print(f"Code 26 means: {describe_substance('26')}")

Functions for allergens/additives
Normalized codes: ('4', '20a', '26')
Expanded gluten group 20: ['20a', '20b', '20c', '20d', '20e', '20f']
Code 26 means: Milk & products thereof


## Meal Plan

The baseline uses student prices, allows any diet category, and does not set optional limits. `solve_meal_plan` returns a status, selected `Meal` objects, total price.

In [5]:
preferences = UserPreferences(role="student", diet_mode="any", max_category_repeat={"vegan":1})
result = solve_meal_plan(meals, preferences)

if result["status"] != "Optimal":
    raise RuntimeError(result.get("reason", f"Solver status: {result['status']}"))

selected_meals = result["selected_meals"]
total_price = result["total_price"]

print(f"Solver status: {result['status']}")
print(f"Total student price: €{total_price:.2f}")

Solver status: Optimal
Total student price: €8.70


## Inspect the selected plan

Review the selected meals with their price, diet classification, and climate rating.

In [6]:
print(f"{'Date':<12}  {'Kind':<9}  {'Price':>8}  {'Diet':<12}  {'Climate':<7}  Meal")
for meal in selected_meals:
    price = meal.price_for(preferences.role)
    print(
        f"{meal.date.isoformat():<12}  {meal.kind.value:<9}  €{price:>6.2f}  "
        f"{meal.diet_category.value:<12}  {meal.climate_rating or '-':<7}  {meal.name}"
    )

print("\nStudent price by selected meal")
for meal in selected_meals:
    price = meal.price_for(preferences.role)
    bar = "#" * round(price * 3)
    print(f"{meal.name[:32]:<32} {bar} €{price:.2f}")

Date          Kind          Price  Diet          Climate  Meal
2026-09-29    main       €  2.90  meat          B        Bratwurst
2026-09-30    main       €  3.90  unknown       A        Salatteller  EH To-Go Mozzarella 
2026-10-01    main       €  1.90  vegan         A        Pommes frites frittiert EH

Student price by selected meal
Bratwurst                        ######### €2.90
Salatteller  EH To-Go Mozzarella ############ €3.90
Pommes frites frittiert EH       ###### €1.90
